# 05 Dashboard data

Builds the tables the Power BI report reads. It is a small star schema: two dimensions (date, package) and fact tables at the grain each page needs. Everything lands in `data/processed/` as CSV, and can also go to BigQuery.

| Table | Grain | Feeds |
|---|---|---|
| dim_date | one row per day | every page (slicers, month axis) |
| dim_package | one row per package | every page |
| fact_daily_package | package x day | Overview, Energy efficiency |
| fact_monthly_package | package x month | Overview trend, Downtime trend |
| fact_downtime_events | one row per stop | Downtime analysis |
| fact_daily_plant | day | Overview (LIN tank, shortfall) |
| fact_anomaly_daily | package x day | Anomaly alerts |
| fact_alert_episodes | one row per alert | Anomaly alerts |

Run notebook 04 first; it writes the anomaly scores this notebook picks up.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from plant_style import (apply_style, save, load, PACKAGE_COLOURS, CAUSE_COLOURS, ALERT, INK_2,
                         MODE_NAMES, RAW, PROCESSED, SPLIT_VALID, SPLIT_TEST)

apply_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

daily = load("daily_package_summary")
downtime = load("downtime_events")
plant = load("daily_plant_summary")
risk = pd.read_csv(PROCESSED / "anomaly_scores_daily.csv", parse_dates=["date"])
episodes = pd.read_csv(PROCESSED / "alert_episodes.csv", parse_dates=["start", "end"])

## Dimensions

In [2]:
dates = pd.DataFrame({"date": pd.date_range(daily.date.min(), daily.date.max(), freq="D")})
dates["year"] = dates.date.dt.year
dates["quarter"] = "Q" + dates.date.dt.quarter.astype(str)
dates["month_start"] = dates.date.dt.to_period("M").dt.to_timestamp()
dates["month_name"] = dates.date.dt.strftime("%b %Y")
dates["month_sort"] = dates.date.dt.year * 100 + dates.date.dt.month
dates["season"] = np.where(dates.date.dt.month.isin([11, 12, 1, 2, 3]), "Dry", "Rainy")
dates["dataset_period"] = np.where(dates.date < SPLIT_VALID, "Train",
                                   np.where(dates.date < SPLIT_TEST, "Validation", "Test"))

packages = pd.DataFrame({
    "package_id": ["NGU-A", "NGU-B", "NGU-C"],
    "tag_series": ["1xx", "2xx", "3xx"],
    "design_rate_nm3h": [1000, 1000, 1000],
    "turnaround_month": ["January", "April", "July"],
    "note": ["Reference package", "Commissioned with NGU-A", "Oldest package, highest fouling rate"],
})
packages

,package_id,tag_series,design_rate_nm3h,turnaround_month,note
0,NGU-A,1xx,1000,January,Reference package
1,NGU-B,2xx,1000,April,Commissioned with NGU-A
2,NGU-C,3xx,1000,July,"Oldest package, highest fouling rate"


## Facts

In [3]:
fact_daily = daily.merge(risk[["date", "package_id", "max_risk_ratio", "alert_hours", "risk_level"]],
                         on=["date", "package_id"], how="left")
fact_daily["failure_mode"] = fact_daily.failure_mode.map(MODE_NAMES).fillna("None")
fact_daily["n2_production_knm3"] = (fact_daily.n2_production_nm3 / 1000).round(2)
fact_daily["power_consumption_mwh"] = (fact_daily.power_consumption_kwh / 1000).round(3)

fact_events = downtime.copy()
fact_events["failure_mode"] = fact_events.failure_mode.map(MODE_NAMES).fillna(
    fact_events.failure_mode.str.replace("_", " ").str.capitalize())
fact_events["date"] = fact_events.trip_time.dt.normalize()
fact_events["onset"] = np.where(fact_events.failure_mode.isin(
    ["Expander bearing", "Compressor vibration", "PPU breakthrough", "Cooling water loss"]), "Gradual",
    np.where(fact_events.event_type.eq("Planned"), "Planned", "Sudden"))

m = daily.assign(month_start=daily.date.dt.to_period("M").dt.to_timestamp())
fact_monthly = m.groupby(["month_start", "package_id"]).agg(
    days=("date", "size"),
    running_hours=("running_hours", "sum"),
    planned_downtime_hrs=("planned_downtime_hrs", "sum"),
    unplanned_downtime_hrs=("unplanned_downtime_hrs", "sum"),
    n2_production_nm3=("n2_production_nm3", "sum"),
    n2_vented_nm3=("n2_vented_nm3", "sum"),
    power_consumption_kwh=("power_consumption_kwh", "sum"),
).reset_index()
fact_monthly["availability_pct"] = 100 * fact_monthly.running_hours / (24 * fact_monthly.days - fact_monthly.planned_downtime_hrs)
made = fact_monthly.n2_production_nm3 + fact_monthly.n2_vented_nm3
fact_monthly["performance_pct"] = 100 * made / (1000 * fact_monthly.running_hours)
fact_monthly["quality_pct"] = 100 * fact_monthly.n2_production_nm3 / made
fact_monthly["oee_pct"] = fact_monthly.availability_pct * fact_monthly.performance_pct * fact_monthly.quality_pct / 1e4
fact_monthly["energy_intensity_kwh_per_nm3"] = fact_monthly.power_consumption_kwh / fact_monthly.n2_production_nm3
trips_m = (fact_events[fact_events.event_type.eq("Unplanned")]
           .assign(month_start=lambda d: d.trip_time.dt.to_period("M").dt.to_timestamp())
           .groupby(["month_start", "package_id"]).size().rename("unplanned_trips"))
fact_monthly = fact_monthly.merge(trips_m, on=["month_start", "package_id"], how="left").fillna({"unplanned_trips": 0})
num = fact_monthly.select_dtypes("number").columns
fact_monthly[num] = fact_monthly[num].round(3)

fact_risk = risk.copy()
fact_episodes = episodes.copy()
fact_episodes["duration_h"] = ((fact_episodes.end - fact_episodes.start) / pd.Timedelta(hours=1) + 1).astype(int)
fact_episodes["date"] = fact_episodes.start.dt.normalize()

## Checks before export

In [4]:
assert np.isclose(fact_monthly.n2_production_nm3.sum(), daily.n2_production_nm3.sum())
assert np.isclose(fact_monthly.unplanned_downtime_hrs.sum(), daily.unplanned_downtime_hrs.sum())
assert fact_daily.duplicated(["date", "package_id"]).sum() == 0
assert set(fact_daily.package_id) == set(packages.package_id)
assert fact_daily.date.isin(dates.date).all()
print("Totals reconcile; keys are unique and every fact row has a date and package.")

Totals reconcile; keys are unique and every fact row has a date and package.


In [5]:
tables = {
    "dim_date": dates,
    "dim_package": packages,
    "fact_daily_package": fact_daily,
    "fact_monthly_package": fact_monthly,
    "fact_downtime_events": fact_events,
    "fact_daily_plant": plant,
    "fact_anomaly_daily": fact_risk,
    "fact_alert_episodes": fact_episodes,
}
PROCESSED.mkdir(parents=True, exist_ok=True)
for name, df in tables.items():
    df.to_csv(PROCESSED / f"{name}.csv", index=False)
    print(f"{name:24s} {len(df):>6,d} rows  {df.shape[1]:>2d} columns")

dim_date                    730 rows   8 columns
dim_package                   3 rows   5 columns
fact_daily_package        2,190 rows  24 columns
fact_monthly_package         72 rows  15 columns
fact_downtime_events        111 rows  12 columns
fact_daily_plant            730 rows  13 columns
fact_anomaly_daily        2,190 rows   7 columns
fact_alert_episodes         108 rows  11 columns


## Headline numbers for the Overview page (latest full month)

In [6]:
last = fact_monthly.month_start.max()
lm = fact_monthly[fact_monthly.month_start.eq(last)]
print(f"Month: {last:%B %Y}")
print(f"N2 to header: {lm.n2_production_nm3.sum()/1e6:.2f} million Nm3")
print(f"Unplanned downtime: {lm.unplanned_downtime_hrs.sum():.0f} h across {int(lm.unplanned_trips.sum())} trips")
print(f"Energy intensity: {lm.power_consumption_kwh.sum()/lm.n2_production_nm3.sum():.3f} kWh/Nm3")
lm[["package_id", "oee_pct", "energy_intensity_kwh_per_nm3", "unplanned_trips"]].round(3)

Month: September 2026
N2 to header: 1.95 million Nm3
Unplanned downtime: 132 h across 4 trips
Energy intensity: 0.322 kWh/Nm3


,package_id,oee_pct,energy_intensity_kwh_per_nm3,unplanned_trips
69,NGU-A,94.183,0.316,1.0
70,NGU-B,93.865,0.317,1.0
71,NGU-C,82.262,0.336,2.0


## Load to BigQuery (optional)

Same switch as notebook 01. The SQL views in `sql/` run against these tables.

In [7]:
LOAD_TO_BIGQUERY = False
GCP_PROJECT = "your-gcp-project-id"
BQ_DATASET = "nitrochem_ngu"

if LOAD_TO_BIGQUERY:
    from load_to_bigquery import load_folder
    load_folder(PROCESSED, GCP_PROJECT, BQ_DATASET)
else:
    print("Skipped BigQuery load.")

Skipped BigQuery load.
